In [ ]:
from time import sleep as delay
from project.el5600 import project
if "ic" in globals():
    ic.close()
ic = project(device="el5600", revision="a1", emulator="cp2112", logging=False, is_gui=False)

from phy.multimeter import siglent_sdm3055_auto
from phy.power_supply import keithley_2460, rigol_dp821a, keysight_e36232a
from phy.scope import tektronix_dpo4104b_usb
from phy.eloader import sdl1020x
from phy.relay_16ch import relay_box
# from phy.chamber_th3 import chamber

%matplotlib tk
from interface.docs.output_chart import plot
from interface.cui_colors import color
from interface.i2c_bridge.tca9548a import tca9548
from interface.docs.output_excel import excel_frame, style
from interface.cui_logger import logger as log

from time import sleep as delay
import numpy as np

chart = plot()

dm = siglent_sdm3055_auto()
ps = rigol_dp821a()
ks = keysight_e36232a()
kt = keithley_2460()
ld = sdl1020x()
sc = tektronix_dpo4104b_usb()

# dma = agilent_34401a("COM5")
relay = relay_box(i2c_h=ic)
# tc = chamber(port=3)
# mux = tca9548(ic.i2c_h, 0x70)


# ==================================
def disable_all_ps(kt=kt, ps=ps):
    kt.disable
    ks.disable
    ld.disable
    ps.ch1.disable
    ps.ch2.disable
# ==================================


# ----------------------------------------------------------------------------
print("██████  ██ ███████  ██████  ██████  ███    ██ ███    ██ ███████  ██████ ████████     ██       ██████   █████  ██████  ")
print("██   ██ ██ ██      ██      ██    ██ ████   ██ ████   ██ ██      ██         ██        ██      ██    ██ ██   ██ ██   ██ ")
print("██   ██ ██ ███████ ██      ██    ██ ██ ██  ██ ██ ██  ██ █████   ██         ██        ██      ██    ██ ███████ ██   ██ ")
print("██   ██ ██      ██ ██      ██    ██ ██  ██ ██ ██  ██ ██ ██      ██         ██        ██      ██    ██ ██   ██ ██   ██ ")
print("██████  ██ ███████  ██████  ██████  ██   ████ ██   ████ ███████  ██████    ██        ███████  ██████  ██   ██ ██████ ")
# ----------------------------------------------------------------------------

# Test A1

KT : ---> VDD3V3 (for current measurement)

In [ ]:
temperature = "N40C"

log.output_set_filename(f"[{temperature}_1] IQ_LPM_VDD3V3, VDD3V3_UVLO, HYS")
log.output_csv(["VDD3V3 (V)", "I_VDD3V3 (uA)", "Register_00h"])

In [ ]:
disable_all_ps()
relay.init_channels

v_start  = 1.5
v_finish = 3.61

list_temp = list(np.arange(v_start, v_finish, 0.1))
list_vdd  = [round(num, 1) for num in list_temp]

# --------------------------------------------
ps.ch1.cfg_all = 5, 1 # relay
ps.ch1.enable

# --------------------------------------------
kt.cfg_all = list_vdd[0], 0.01 # init
kt.enable
delay(1)

In [ ]:
# ----------------------------------------------------------------------------

for vdd in list_vdd:
    
    kt.vset = vdd
    i_vdd = round(kt.current * 1e+6,6) # uA unit
    
    try:
        reg_00h = ic.read_byte(0x00)
    except:
        reg_00h = "NACK"
    
    ret = [vdd, i_vdd, reg_00h]
    log.output_csv(ret)
    
    print(f"VDD3V3={vdd},  I_VDD3V3={i_vdd:.06f}uA, Register_00h={reg_00h}")

# ----------------------------------------------------------------------------

for vdd in reversed(list_vdd):
    
    kt.vset = vdd
    i_vdd = round(kt.current * 1e+6,6) # uA unit
    
    try:
        reg_00h = ic.read_byte(0x00)
    except:
        reg_00h = "NACK"
    
    ret = [vdd, i_vdd, reg_00h]
    log.output_csv(ret)
    
    print(f"VDD3V3={vdd},  I_VDD3V3={i_vdd:.06f}uA, Register_00h={reg_00h}")
    
# ----------------------------------------------------------------------------

relay.init_channels
disable_all_ps()

# Debugging

In [ ]:
sc.ch1.vertical_scale = 5
sc.ch1.vertical_grid = -2
sc.ch2.vertical_scale = 5
sc.ch2.vertical_grid = -3
sc.ch4.vertical_scale = 0.5
sc.ch4.vertical_grid = -4